In [2]:
import time
import random
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torchvision.transforms import v2
from torch.utils.tensorboard import SummaryWriter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

I0000 00:00:1792701770.670502  161378 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1792701770.956190  161378 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792701772.300306  161378 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


cuda


# 1. Same network as in the PyTorch CNN tutorial

In [3]:
transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 4

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x


net = Net()

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)


for epoch in range(2):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 2000 == 1999:    # print every 2000 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
            running_loss = 0.0

print('Finished Training')


correct = 0
total = 0
# since we're not training, we don't need to calculate the gradients for our outputs
with torch.no_grad():
    for data in testloader:
        images, labels = data
        # calculate outputs by running images through the network
        outputs = net(images)
        # the class with the highest energy is what we choose as prediction
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on the 10000 test images: {100 * correct // total} %')


# prepare to count predictions for each class
correct_pred = {classname: 0 for classname in classes}
total_pred = {classname: 0 for classname in classes}

# again no gradients needed
with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = net(images)
        _, predictions = torch.max(outputs, 1)
        # collect the correct predictions for each class
        for label, prediction in zip(labels, predictions):
            if label == prediction:
                correct_pred[classes[label]] += 1
            total_pred[classes[label]] += 1


# print accuracy for each class
for classname, correct_count in correct_pred.items():
    accuracy = 100 * float(correct_count) / total_pred[classname]
    print(f'Accuracy for class: {classname:5s} is {accuracy:.1f} %')

[1,  2000] loss: 2.185
[1,  4000] loss: 1.933
[1,  6000] loss: 1.687
[1,  8000] loss: 1.583
[1, 10000] loss: 1.533
[1, 12000] loss: 1.477
[2,  2000] loss: 1.378
[2,  4000] loss: 1.374
[2,  6000] loss: 1.374
[2,  8000] loss: 1.336
[2, 10000] loss: 1.306
[2, 12000] loss: 1.280
Finished Training
Accuracy of the network on the 10000 test images: 48 %
Accuracy for class: plane is 60.5 %
Accuracy for class: car   is 22.6 %
Accuracy for class: bird  is 39.1 %
Accuracy for class: cat   is 28.3 %
Accuracy for class: deer  is 26.0 %
Accuracy for class: dog   is 43.6 %
Accuracy for class: frog  is 64.2 %
Accuracy for class: horse is 62.3 %
Accuracy for class: ship  is 77.6 %
Accuracy for class: truck is 60.3 %


# 2. Architecture change

In [4]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 8, 3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)
        self.conv3 = nn.Conv2d(16, 32, 3, padding=1)
        self.conv4 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 8 * 8, 4096)
        self.fc2 = nn.Linear(4096, 1000)
        self.fc3 = nn.Linear(1000, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(F.relu(self.conv2(x)))
        x = F.relu(self.conv3(x))
        x = self.pool(F.relu(self.conv4(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

## Training and evaluation helpers
`train()` logs the training loss to TensorBoard (see section 4), `evaluate()` reports top-1 and top-3 accuracy (see section 5).

In [ ]:
def train(net, device, writer, epochs=2, lr=0.001, loader=None, log_every=2000):
    loader = loader or trainloader
    net.to(device)
    net.train()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(net.parameters(), lr=lr, momentum=0.9)
    start = time.perf_counter()
    for epoch in range(epochs):
        running_loss = 0.0
        for i, (inputs, labels) in enumerate(loader, 0):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = net(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            if i % log_every == log_every - 1:
                print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / log_every:.3f}')
                writer.add_scalar('training loss', running_loss / log_every,
                                  (epoch * len(loader) + i) * loader.batch_size)
                running_loss = 0.0
    if device.type == 'cuda':
        torch.cuda.synchronize()
    duration = time.perf_counter() - start
    writer.flush()
    print(f'Finished Training on {device} in {duration:.1f} s')
    return duration

## Training on the CPU (baseline for the speedup comparison)

In [6]:
torch.manual_seed(0)
net_cpu = Net()

# passiv aufgabe 4
cpu_writer = SummaryWriter('runs/cifar10/net_cpu')
cpu_writer.add_graph(net_cpu, images)
cpu_time = train(net_cpu, torch.device('cpu'), cpu_writer)
cpu_writer.close()

/home/debian/jupyter-env/lib/python3.14/site-packages/torch/jit/_trace.py:1000: FutureWarning: `torch.jit.trace` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/home/debian/jupyter-env/lib/python3.14/site-packages/torch/jit/_trace.py:1139: FutureWarning: `torch.jit.trace_method` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


[1,  2000] loss: 2.286
[1,  4000] loss: 1.959
[1,  6000] loss: 1.750
[1,  8000] loss: 1.636
[1, 10000] loss: 1.514
[1, 12000] loss: 1.441
[2,  2000] loss: 1.302
[2,  4000] loss: 1.242
[2,  6000] loss: 1.200
[2,  8000] loss: 1.141
[2, 10000] loss: 1.106
[2, 12000] loss: 1.053
Finished Training on cpu in 1255.7 s


# 3. Train on GPU

In [7]:
torch.manual_seed(0)
net = Net()

# passiv aufgabe 4
gpu_writer = SummaryWriter('runs/cifar10/net_gpu')
gpu_writer.add_graph(net, images)
gpu_time = train(net, device, gpu_writer)

print(f'CPU: {cpu_time:.1f} s')
print(f'GPU: {gpu_time:.1f} s')
print(f'Speedup: {cpu_time / gpu_time:.2f}x')

[1,  2000] loss: 2.287
[1,  4000] loss: 1.960
[1,  6000] loss: 1.752
[1,  8000] loss: 1.637
[1, 10000] loss: 1.514
[1, 12000] loss: 1.443
[2,  2000] loss: 1.304
[2,  4000] loss: 1.244
[2,  6000] loss: 1.201
[2,  8000] loss: 1.142
[2, 10000] loss: 1.111
[2, 12000] loss: 1.049
Finished Training on cuda in 105.7 s
CPU: 1255.7 s
GPU: 105.7 s
Speedup: 11.88x


# 5. Top 3 accuracy

In [10]:
def evaluate(net, device, k=3):
    net.to(device)
    net.eval()
    top1 = topk = total = 0
    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = net(images)
            top = outputs.topk(k, dim=1).indices
            top1 += (top[:, 0] == labels).sum().item()
            topk += (top == labels.unsqueeze(1)).any(dim=1).sum().item()
            total += labels.size(0)
    print(f'Top-1 accuracy: {100 * top1 / total:.2f} %')
    print(f'Top-{k} accuracy: {100 * topk / total:.2f} %')
    return topk / total

evaluate(net, device)

Top-1 accuracy: 63.98 %
Top-3 accuracy: 89.62 %


0.8962

# 6. Top-3 errors in TensorBoard
Random test images whose true label is not among the top three predictions, written to TensorBoard.

In [11]:
def top3_errors(net, device, n=5):
    net.to(device)
    net.eval()
    errors = []
    with torch.no_grad():
        for images, labels in testloader:
            outputs = net(images.to(device)).cpu()
            top = outputs.topk(3, dim=1).indices
            for img, label, pred in zip(images, labels, top):
                if label not in pred:
                    errors.append((img, label.item(), pred.tolist()))
    return random.sample(errors, n)


samples = top3_errors(net, device)

fig = plt.figure(figsize=(15, 4))
for idx, (img, label, pred) in enumerate(samples):
    ax = fig.add_subplot(1, 5, idx + 1, xticks=[], yticks=[])
    ax.imshow(np.transpose((img / 2 + 0.5).numpy(), (1, 2, 0)))
    ax.set_title(f'true: {classes[label]}\npred: {", ".join(classes[p] for p in pred)}')
gpu_writer.add_figure('top-3 errors', fig)
gpu_writer.add_images('top-3 error images', torch.stack([s[0] for s in samples]) / 2 + 0.5)
gpu_writer.add_text('top-3 error labels', '  \n'.join(
    f'{i}: true={classes[l]}, pred={[classes[p] for p in pr]}' for i, (_, l, pr) in enumerate(samples)))
gpu_writer.close()

## Precision-recall curves per class

In [ ]:
probs, targets = [], []
net.eval()
with torch.no_grad():
    for images, labels in testloader:
        probs.append(F.softmax(net(images.to(device)), dim=1).cpu())
        targets.append(labels)
probs, targets = torch.cat(probs), torch.cat(targets)

pr_writer = SummaryWriter('runs/cifar10/net_gpu')
for i, classname in enumerate(classes):
    pr_writer.add_pr_curve(classname, targets == i, probs[:, i], global_step=0)
pr_writer.close()


# 4. TensorBoard

In [14]:
%load_ext tensorboard
#%tensorboard --logdir runs/cifar10
from IPython.display import IFrame
IFrame('http://localhost:6006', width='100%', height=800)


The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard
